In [ ]:
import numpy as np

def munkres_verbose(cost):
    matrix = cost.copy().astype(float)
    n = matrix.shape[0]

    print("\n===== INITIAL MATRIX (DELIVERY COST MATRIX) =====")
    print(matrix)

    # Step 1: Row reduction
    print("\n===== ROW REDUCTION =====")
    for i in range(n):
        mv = matrix[i].min()
        print(f"Row {i} min = {mv}")
        matrix[i] -= mv

    print("\nMatrix after row reduction:")
    print(matrix)

    # Step 2: Column reduction
    print("\n===== COLUMN REDUCTION =====")
    for j in range(n):
        mv = matrix[:, j].min()
        print(f"Col {j} min = {mv}")
        matrix[:, j] -= mv

    print("\nMatrix after column reduction:")
    print(matrix)

    # Prepare structures
    starred = np.zeros((n, n), dtype=bool)
    primed = np.zeros((n, n), dtype=bool)
    row_cov = np.zeros(n, dtype=bool)
    col_cov = np.zeros(n, dtype=bool)

    # Star initial zeros
    for i in range(n):
        for j in range(n):
            if matrix[i, j] == 0 and not row_cov[i] and not col_cov[j]:
                starred[i, j] = True
                row_cov[i] = True
                col_cov[j] = True

    row_cov[:] = False
    col_cov[:] = False

    print("\nInitial starred zeros (1 = starred):")
    print(starred.astype(int))

    # Cover columns containing starred zeros
    for j in range(n):
        if starred[:, j].any():
            col_cov[j] = True

    print("Initially covered columns:", np.where(col_cov)[0].tolist())

    iteration = 1

    while True:

        print(f"\n===== ITERATION {iteration} =====")

        if col_cov.sum() == n:
            print("All columns covered — Optimal Driver → Route assignment found.")
            break

        def find_uncovered_zero():
            for i in range(n):
                if not row_cov[i]:
                    for j in range(n):
                        if not col_cov[j] and matrix[i, j] == 0:
                            return (i, j)
            return None

        z = find_uncovered_zero()

        while z is not None:

            i, j = z
            primed[i, j] = True

            print(f"Primed zero at ({i},{j})")

            star_col = np.where(starred[i])[0]

            if star_col.size == 0:

                path = [(i, j)]

                while True:

                    starred_row = np.where(starred[:, path[-1][1]])[0]

                    if starred_row.size == 0:
                        break

                    starred_row = int(starred_row[0])

                    path.append((starred_row, path[-1][1]))

                    prime_col = np.where(primed[path[-1][0]])[0]
                    prime_col = int(prime_col[0])

                    path.append((path[-1][0], prime_col))

                print("Augmenting path:", path)

                for r, c in path:
                    starred[r, c] = not starred[r, c]

                primed[:, :] = False
                row_cov[:] = False
                col_cov[:] = False

                for col in range(n):
                    if starred[:, col].any():
                        col_cov[col] = True

                print("Starred matrix now:")
                print(starred.astype(int))

                break

            else:

                sc = int(star_col[0])

                row_cov[i] = True
                col_cov[sc] = False

                print(f"Row {i} covered; Column {sc} uncovered")

                z = find_uncovered_zero()

        if find_uncovered_zero() is None:

            min_uncovered = float('inf')

            for ii in range(n):
                if not row_cov[ii]:
                    for jj in range(n):
                        if not col_cov[jj] and matrix[ii, jj] < min_uncovered:
                            min_uncovered = matrix[ii, jj]

            if min_uncovered == float('inf'):
                min_uncovered = 0

            print("Minimum uncovered value:", min_uncovered)

            for ii in range(n):
                for jj in range(n):

                    if not row_cov[ii] and not col_cov[jj]:
                        matrix[ii, jj] -= min_uncovered

                    elif row_cov[ii] and col_cov[jj]:
                        matrix[ii, jj] += min_uncovered

            print("Matrix after adjustment:")
            print(matrix)

        iteration += 1

    assignment = []

    for i in range(n):
        j_idx = np.where(starred[i])[0]

        if j_idx.size > 0:
            assignment.append((i, int(j_idx[0])))

    return assignment


# --------------------------------
# USER INPUT PART (FRESHDRINKS)
# --------------------------------

rows = int(input("Enter number of Delivery Drivers: "))
cols = int(input("Enter number of Delivery Routes: "))

print(f"\nEnter delivery cost matrix row by row ({rows} x {cols}):")

user_matrix = []

for i in range(rows):
    row = list(map(float, input(f"Driver {i+1}: ").split()))
    user_matrix.append(row)

user_matrix = np.array(user_matrix)

# Balance matrix (make square)
n = max(rows, cols)

balanced = np.zeros((n, n))
balanced[:rows, :cols] = user_matrix

print("\nBalanced Delivery Cost Matrix:")
print(balanced)

assign = munkres_verbose(balanced)

print("\n===== FINAL ASSIGNMENT (DRIVER → ROUTE) =====")

total = 0

for r, c in assign:

    if r < rows and c < cols:

        print(
            f"Driver {r+1} -> Route {c+1} | Cost = Rs. {user_matrix[r, c]}"
        )

        total += user_matrix[r, c]

print("✅ Total Minimum Delivery Cost = Rs.", total)